# Random Forest v2.2 — DNS Attack Detection

## Why Random Forest?
- Naturally resistant to overfitting on dominant features (unlike XGBoost)
- Each tree sees a random subset of features → `bwd_packets_per_sec` cannot dominate
- No gradient boosting means less sensitivity to class imbalance
- Good baseline to compare against XGBoost and LightGBM

## Same Pipeline as XGBoost v2.2
- Log-transform 8 skewed features
- `class_weight='balanced'` (RF equivalent of `scale_pos_weight`)
- F-beta threshold tuning (beta=2.0 → prioritise attack recall)
- Balance report showing both BENIGN and ATTACK recall

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print('[OK] Libraries loaded')

## 1. Load Data

In [ ]:
FILE_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

print(f'[INFO] Loading: {FILE_PATH}')
df = pd.read_csv(FILE_PATH)
print(f'[OK]  Rows: {len(df):,}   Cols: {len(df.columns)}')
print(f'\nClass Distribution:')
print(df['label'].value_counts())
df.head(3)

## 2. Preprocessing + Log-Transform

In [ ]:
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')

df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels: {dict(df["label"].value_counts())}')

PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
le_proto = LabelEncoder()
le_proto.fit(PROTOCOL_CLASSES)
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = le_proto.transform(df['protocol'])

# Log-transform skewed features — prevents streaming traffic from looking like DDoS
LOG_FEATURES = [
    'bwd_packets_per_sec',
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'fwd_packets_per_sec',
    'dns_queries_per_second',
    'total_fwd_packets',
    'total_bwd_packets',
    'dns_amplification_factor',
]
print('\n[LOG-TRANSFORM] Compressing skewed features...')
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))
        print(f'  log1p({col})')

print('\n[OK] Preprocessing complete')

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
print(f'Train: BENIGN={n_benign:,}  ATTACK={n_attack:,}  ratio={n_attack/n_benign:.2f}')
print(f'Test:  {X_test.shape}')

## 4. Random Forest Training

### Key RF Parameters
| Parameter | Value | Effect |
|---|---|---|
| `class_weight` | `balanced` | Automatically weights BENIGN vs ATTACK by inverse frequency |
| `n_estimators` | `500` | 500 trees — more stable than 100 |
| `max_features` | `'sqrt'` | Each tree sees sqrt(44)≈6 features — prevents `bwd_pps` dominance |
| `min_samples_leaf` | `5` | Prevents overfitting to small attack clusters |
| `max_depth` | `20` | Prevents very deep trees memorising training data |

In [ ]:
# class_weight options:
#   'balanced'          = auto-weight by inverse class frequency
#   'balanced_subsample'= same but recomputed per tree (better for imbalanced)
#   {0: 1, 1: 2}        = manual: ATTACK is 2x more important
CLASS_WEIGHT = 'balanced'

model = RandomForestClassifier(
    n_estimators      = 500,
    max_depth         = 20,           # Limit depth to prevent overfit
    min_samples_leaf  = 5,            # Min 5 samples at leaf
    min_samples_split = 10,           # Min 10 samples to split a node
    max_features      = 'sqrt',       # sqrt(n_features) per split — key for balance
    class_weight      = CLASS_WEIGHT, # Handle class imbalance
    n_jobs            = -1,           # Use all CPU cores
    random_state      = 42,
    verbose           = 1,
)

print(f'[TRAIN] class_weight={CLASS_WEIGHT}, max_features=sqrt, n_estimators=500')
model.fit(X_train, y_train)
print('[DONE] Training complete.')

## 5. Threshold Tuning (ATTACK Recall Priority)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

# beta=2.0 → attack recall is 4x more important than false alarm rate
BETA = 2.0

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    score = fbeta_score(y_test, y_pred_t, beta=BETA, zero_division=0)
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')

scores = [fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
          for t in thresholds]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='forestgreen')
plt.axvline(best_thresh, color='red',  linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5,         color='gray', linestyle=':',  label='Default=0.50')
plt.xlabel('Threshold'); plt.ylabel(f'F-beta (beta={BETA})')
plt.title('Random Forest — Threshold vs Score'); plt.legend(); plt.show()

## 6. Balance Report

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred, target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'Random Forest Confusion Matrix @ threshold={best_thresh:.2f}')
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' RF BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%')
print(f'===============================')

if attack_recall < 0.85:
    print('\n[ADVICE] Try: class_weight={0:1, 1:2} to boost attack sensitivity')
elif benign_recall < 0.85:
    print('\n[ADVICE] Try: class_weight="balanced" or increase min_samples_leaf')
else:
    print('\n[GOOD] Both recalls above 85%!')

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='forestgreen')
plt.title('Top 20 Feature Importances — Random Forest v2.2')
plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()

bwd_imp = importance.get('bwd_packets_per_sec', 0)
print(f'\n[CHECK] bwd_packets_per_sec = {bwd_imp:.4f}')
if bwd_imp > 0.15:
    print('  [WARN] Still significant. Try reducing max_features to 0.3')
else:
    print('  [GOOD] Well distributed across DNS features.')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':            model,
    'threshold':        best_thresh,
    'log_features':     LOG_FEATURES,
    'protocol_classes': PROTOCOL_CLASSES,
    'model_type':       'random_forest',
    'beta':             BETA,
}

with open('rf_model_v2.2.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] rf_model_v2.2.pkl')
print(f'  threshold    = {best_thresh:.2f}')
print(f'  ATTACK Recall = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall = {benign_recall*100:.1f}%')
print(f'  ROC-AUC       = {auc:.4f}')